In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch_directml

# Tokenization

### 토큰화 (Tokenization): 글자를 숫자로 매핑
- 모델은 "이름"을 바로 배울 수 없으므로, 모든 글자를 숫자로 바꿉니다.
    - 코드 포인트: stoi(글자→숫자), itos(숫자→글자) 사전을 만듭니다.
    - Special Token: 문장의 시작과 끝을 표현하는 .을 0번 인덱스로 할당합니다.

In [2]:
words = open('names.txt', 'r').read().splitlines()

In [3]:
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [ ]:
# (1) 문자 사전 만들기
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

# (2) 입력(xs)과 정답(ys) 데이터셋 생성
xs, ys = [], []
for w in words[:1]:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1, ix2 = stoi[ch1], stoi[ch2]
    xs.append(ix1) # 입력 글자 인덱스
    ys.append(ix2) # 정답 글자 인덱스

# (3) One-hot Encoding (신경망 입력용)
import torch.nn.functional as F
xenc = F.one_hot(xs, num_classes=27).float()

# Broadcasting

- 카운트 행렬($N$)을 확률 행렬($P$)로 바꿀 때, 한 줄(Row)의 합계로 모든 칸을 나눠야 합니다. 이때 행렬의 크기가 달라도 PyTorch가 알아서 맞춰주는 기능
- 핵심: P.sum(1, keepdim=True)를 써서 차원을 유지해야 정확히 한 줄씩 나뉩니다.
- Smoothing: 빈도가 0인 경우 로그 계산 시 $-\infty$가 나오므로, 모든 칸에 $+1$을 해줍니다.

In [ ]:
# (1) 선형 연산 (W는 가중치)
logits = xenc @ W 

# (2) Softmax 과정 (확률로 변환)
counts = logits.exp() # 1. 모든 값을 양수로 (e^x)
# 2. 행별 합계로 나누기 (여기서 Broadcasting 발생!)
probs = counts / counts.sum(1, keepdims=True)

# Negative Log Likelihood

- 모델이 예측한 정답 확률을 극대화하도록 유도하는 손실 함수입니다.
- 왜 로그인가?: 확률은 곱할수록 0에 수렴하므로, 로그를 취해 덧셈으로 바꿉니다.
- 왜 마이너스인가?: 로그 확률은 항상 음수이므로, 우리가 보기 편하게 양수로 바꿉니다.

# (1) 모델이 정답(ys)에 부여한 확률만 쏙 골라내기
# probs[torch.arange(5), ys] -> 정답 위치의 확률값들

# (2) 로그를 씌우고 평균을 내서 마이너스를 붙임
loss = -probs[torch.arange(num), ys].log().mean()